# Generate (n, k, m, P) Training Data with LP-computed m-Heights

Generates diverse (n, k, m, P) samples using an evolutionary search strategy that enriches low-m-height codes while also covering the full height distribution via random scatter.

- **n** = 9 (fixed)
- **k** ∈ {4, 5, 6}
- **m** ∈ {2, ..., n−k}
- **P** is a k × (n−k) matrix of integers

In [19]:
import numpy as np
from scipy.optimize import linprog
from itertools import combinations, product
from multiprocessing import Pool, cpu_count
import pickle
import time
import sys

## 1. LP-based m-Height computation

Exact LP algorithm from the project spec (Theorem 1). Each sample is solved independently, so we parallelize across CPU cores.

In [ ]:
def compute_m_height(n, k, m, P):
    """
    Compute m-height using the simplified LP algorithm from:
    Roth et al., "On the Height Profile of Analog Error-Correcting Codes"
    (Theorem 1 + Figure 1).

    For each m-subset S of [n) (the "free" positions that can be large):
      - Constrain entries at positions S_bar to [-1, 1]:  -1 <= u·g_j <= 1 for j in S_bar
      - For each i in S, maximize u·g_i

    Total LPs: m * C(n, m).
    """
    G = np.hstack([np.eye(k), P])
    g_cols = G.T  # g_cols[j] = j-th column of G (length k)

    h = 0.0
    indices = list(range(n))

    # Enumerate all m-subsets S of [n) — these are the m "free" positions
    for S in combinations(indices, m):
        S_set = set(S)
        S_bar = [j for j in indices if j not in S_set]  # complement, size n-m

        # Constraints on S_bar: -1 <= u · g_j <= 1 for all j in S_bar
        G_Sbar = np.array([g_cols[j] for j in S_bar])  # (n-m, k)
        A_ub = np.vstack([G_Sbar, -G_Sbar])             # (2(n-m), k)
        b_ub = np.ones(2 * (n - m))

        # For each i in S (the free positions), maximize u · g_i
        for i in S:
            c_obj = -g_cols[i]  # negate for minimization
            result = linprog(c_obj, A_ub=A_ub, b_ub=b_ub,
                             bounds=[(None, None)] * k,
                             method='highs',
                             options={'presolve': True,
                                      'dual_feasibility_tolerance': 1e-8,
                                      'primal_feasibility_tolerance': 1e-8})
            if result.success:
                val = -result.fun
                if val > h:
                    h = val
            elif result.status == 3:  # unbounded
                return float('inf')

    return h


def _worker(args):
    """Worker function for multiprocessing."""
    idx, n, k, m, P = args
    height = compute_m_height(n, k, m, P)
    return idx, height


def _search_score_worker(args):
    """Score one candidate generator matrix across all target m values."""
    idx, G, n, k, target_ms = args
    scores = evaluate_G_multi_m(G, n, k, target_ms)
    scalar = float(np.mean([scores[m] for m in target_ms]))
    return idx, {"G": G, "scores": scores, "scalar": scalar}

print(f"LP solver ready. Available CPUs: {cpu_count()}")

## 2. Diverse code search: low-m-height enrichment + random scatter

Generates (n, k, m, P) samples with a **two-part strategy per group**:

- **40% from evolutionary search** (low-m-height enrichment): runs a genetic programming loop to find codes near the theoretical minimum. These cover the rare low-m-height region (heights 2–50) that random sampling almost never reaches.
- **60% random scatter** (full-distribution coverage): random Gaussian + APC P matrices computed in parallel. These restore the right-skewed tail (heights up to millions) that pure search squashes.

**Why both?** The professor's data spans [2, 6.45M] with median 307. Pure search collapses the range to [8, 5186]. Pure random misses the low end. The mix reproduces both extremes.

In [ ]:
# Per-(k,m) Normal sigma values — estimated from reference data as σ = mean|P| / sqrt(2/π)
sigma_map = {
    (4, 2): 37, (4, 3): 38, (4, 4): 40, (4, 5): 51,
    (5, 2): 35, (5, 3): 37, (5, 4): 46,
    (6, 2): 29, (6, 3): 40,
}

APC_FRACTION = 0.20


def min_distance(P_mat, k, n):
    """Compute minimum distance d of the code [I_k | P] via parity check matrix."""
    r = n - k
    H = np.hstack([-P_mat.T, np.eye(r)])
    for t in range(1, n + 1):
        for cols in combinations(range(n), t):
            sub = H[:, list(cols)]
            if np.linalg.matrix_rank(sub, tol=1e-8) < min(t, r):
                return t
    return n + 1


def gen_gaussian_P(k, n, rng_obj, sigma, clip_value=100):
    """Sample P from Normal(0, sigma), rounded to integers, clipped to [-clip, clip]."""
    r = n - k
    return np.clip(
        np.round(rng_obj.normal(0, sigma, size=(k, r))), -clip_value, clip_value
    ).astype(int)


def gen_apc_P(k, n, rng_obj, sigma, clip_value=100):
    """Punctured Analog Permutation Code (Jiang & Zuo NVMW 2024):
    Columns of P are distinct permutations of a single Gaussian base vector.
    """
    from itertools import permutations as _perms
    r = n - k
    base = rng_obj.normal(0, sigma, size=k)
    all_perms = list({tuple(p) for p in _perms(base)})
    rng_obj.shuffle(all_perms)
    cols = [np.array(all_perms[i % len(all_perms)]) for i in range(r)]
    return np.clip(np.round(np.stack(cols, axis=1)), -clip_value, clip_value).astype(int)



# Search helpers

def make_systematic_G(P):
    """Build a systematic generator matrix G = [I_k | P]."""
    P = np.asarray(P, dtype=float)
    k = P.shape[0]
    return np.hstack([np.eye(k), P])


def project_P_to_train_domain(P, sigma, clip_value=100):
    """Project a candidate P into the integer-valued train data domain."""
    P = np.asarray(P, dtype=float)
    scale = sigma / max(np.std(P), 1e-8)
    projected = np.clip(np.round(P * scale), -clip_value, clip_value).astype(int)
    return projected


def build_G_random(n, k, rng, sigma=1.0, clip_value=100, integerize=True):
    """Dense random generator matrix G = [I_k | P]."""
    r = n - k
    P = rng.normal(0.0, sigma, size=(k, r))
    if integerize:
        P = np.clip(np.round(P), -clip_value, clip_value).astype(int)
    return make_systematic_G(P)


def build_G_generalized_repetition(n, k, scale=1.0, clip_value=100, integerize=True):
    """Generalized repetition style G = [I_k | P] with balanced repeats."""
    r = n - k
    if r <= 0:
        return np.eye(k)
    counts = np.zeros(k, dtype=int)
    cols = []
    for _ in range(r):
        i = int(np.argmin(counts))
        col = np.zeros((k,))
        col[i] = 1.0
        cols.append(col)
        counts[i] += 1
    P = np.stack(cols, axis=1)
    if integerize:
        P = project_P_to_train_domain(P, sigma=max(scale, 1.0), clip_value=clip_value)
    else:
        P = scale * P
    return make_systematic_G(P)


def all_perm_columns_from_base(base):
    """Unique permutations of a base vector, returned as column vectors."""
    from itertools import permutations
    cols = []
    seen = set()
    for p in permutations(base):
        if p in seen:
            continue
        seen.add(p)
        cols.append(np.array(p, dtype=float))
    return cols


def build_G_permutation(n, k, rng, puncture=False, scale=1.0, clip_value=100, integerize=True):
    """Permutation-style construction: choose columns from permutations of one base vector."""
    r = n - k
    if r <= 0:
        return np.eye(k)
    base = np.arange(1, k + 1, dtype=float)
    rng.shuffle(base)
    base = scale * base / np.linalg.norm(base)
    perm_cols = all_perm_columns_from_base(base)
    rng.shuffle(perm_cols)
    if puncture:
        P_cols = perm_cols[:r]
    else:
        P_cols = []
        for i in range(r):
            P_cols.append(perm_cols[i % len(perm_cols)])
    P = np.stack(P_cols, axis=1)
    if integerize:
        P = project_P_to_train_domain(P, sigma=max(scale, 1.0), clip_value=clip_value)
    return make_systematic_G(P)


def mutate_G(G, rng, sigma=1.0, clip_value=100, integerize=True):
    """Gaussian perturbation on P only, preserving systematic form [I|P]."""
    k, n = G.shape
    P = G[:, k:].astype(float).copy()
    P += rng.normal(0.0, sigma, size=P.shape)
    if integerize:
        P = np.clip(np.round(P), -clip_value, clip_value).astype(int)
    return make_systematic_G(P)


def crossover_G(G1, G2, rng, clip_value=100, integerize=True):
    """Column-wise crossover on P blocks."""
    k, n = G1.shape
    r = n - k
    P1 = G1[:, k:]
    P2 = G2[:, k:]
    mask = rng.integers(0, 2, size=r).astype(bool)
    P3 = np.where(mask[np.newaxis, :], P1, P2)
    if integerize:
        P3 = np.clip(np.round(P3), -clip_value, clip_value).astype(int)
    return make_systematic_G(P3)


def evaluate_G_multi_m(G, n, k, target_ms):
    """Return dict m->h_m for requested m values."""
    P = G[:, k:]
    out = {}
    for m in target_ms:
        out[m] = compute_m_height(n, k, m, P)
    return out


def dominates(a, b, target_ms):
    """Pareto dominance for minimization."""
    le_all = all(a[m] <= b[m] for m in target_ms)
    lt_any = any(a[m] < b[m] for m in target_ms)
    return le_all and lt_any


def pareto_front(scored, target_ms):
    """Filter non-dominated candidates."""
    front = []
    for i, ci in enumerate(scored):
        dominated = False
        for j, cj in enumerate(scored):
            if i == j:
                continue
            if dominates(cj["scores"], ci["scores"], target_ms):
                dominated = True
                break
        if not dominated:
            front.append(ci)
    return front


In [ ]:
# Evolutionary search
def search_codes(
    n,
    k,
    target_ms,
    seed=2026,
    pop_size=24,
    generations=8,
    n_mut=24,
    n_cross=16,
    sigma_init=1.0,
    sigma_mut=0.06,
    n_workers=None,
    chunksize=1,
    clip_value=100,
    integerize=True,
    verbose=False,
):
    rng_local = np.random.default_rng(seed)
    target_ms = tuple(sorted(target_ms))
    if n_workers is None:
        n_workers = max(1, min(cpu_count(), pop_size))
    else:
        n_workers = max(1, min(cpu_count(), n_workers))

    # 1) Initialize pool with mixed families
    init_G = []
    n_each = max(2, pop_size // 4)
    for _ in range(n_each):
        init_G.append(build_G_random(n, k, rng_local, sigma=sigma_init, clip_value=clip_value, integerize=integerize))
    for _ in range(n_each):
        init_G.append(build_G_permutation(n, k, rng_local, puncture=False, scale=sigma_init,
                                          clip_value=clip_value, integerize=integerize))
    for _ in range(n_each):
        init_G.append(build_G_permutation(n, k, rng_local, puncture=True, scale=sigma_init,
                                          clip_value=clip_value, integerize=integerize))
    while len(init_G) < pop_size - 1:
        init_G.append(build_G_random(n, k, rng_local, sigma=sigma_init, clip_value=clip_value, integerize=integerize))
    init_G.append(build_G_generalized_repetition(n, k, scale=sigma_init,
                                                 clip_value=clip_value, integerize=integerize))

    def score_pool(pool_G, pool_obj):
        if not pool_G:
            return []
        work_items = [(idx, G, n, k, target_ms) for idx, G in enumerate(pool_G)]
        scored_local = [None] * len(pool_G)
        for idx, scored_entry in pool_obj.imap_unordered(
            _search_score_worker, work_items, chunksize=chunksize,
        ):
            scored_local[idx] = scored_entry
        return scored_local

    with Pool(n_workers) as score_pool_workers:
        scored = score_pool(init_G, score_pool_workers)
        scored.sort(key=lambda x: x["scalar"])
        scored = scored[:pop_size]

        for gen in range(1, generations + 1):
            parents = [x["G"] for x in scored]
            children = []
            for _ in range(n_mut):
                p = parents[rng_local.integers(0, len(parents))]
                children.append(mutate_G(p, rng_local, sigma=sigma_mut,
                                         clip_value=clip_value, integerize=integerize))
            for _ in range(n_cross):
                i1 = rng_local.integers(0, len(parents))
                i2 = rng_local.integers(0, len(parents))
                children.append(crossover_G(parents[i1], parents[i2], rng_local,
                                            clip_value=clip_value, integerize=integerize))
            scored_children = score_pool(children, score_pool_workers)
            scored = scored + scored_children
            scored.sort(key=lambda x: x["scalar"])
            scored = scored[:pop_size]

    front = pareto_front(scored, target_ms)
    front.sort(key=lambda x: x["scalar"])
    return scored, front

In [ ]:
SEARCH_N              = 9
SEARCH_SAMPLE_TARGET  = 100_000
SEARCH_WORKERS        = min(31, cpu_count())
SEARCH_BASE_SEED      = 2026
SEARCH_POP_SIZE       = 300          # 3x larger for deeper search
SEARCH_GENERATIONS    = 10           # 10 gens to find near-optimal codes
SEARCH_MAX_RUNS       = 20           # more runs to collect enough unique samples
SEARCH_CLIP_VALUE     = 100
SEARCH_FRACTION       = 0.40         # 40% search (more budget for low-end coverage)
LOG_BATCH             = 1_000

# Ensure configs and sigma_map are defined even if section 1 wasn't run in this session
_configs = []
for _k in [4, 5, 6]:
    for _m in range(2, SEARCH_N - _k + 1):
        _configs.append((_k, _m))
configs = _configs

sigma_map = {
    (4, 2): 37, (4, 3): 38, (4, 4): 40, (4, 5): 51,
    (5, 2): 35, (5, 3): 37, (5, 4): 46,
    (6, 2): 29, (6, 3): 40,
}

APC_FRACTION = 0.20

# Groups with largest low-end gap get extra search generations
_extra_gens = {
    (4, 2): 5,   # gap: 4x
    (4, 3): 8,   # gap: 13x
    (4, 4): 8,   # gap: 12x
    (4, 5): 10,  # gap: 25x
    (6, 2): 8,   # gap: 18x
}

search_configs = sorted(configs)
base_group_target = SEARCH_SAMPLE_TARGET // len(search_configs)
extra             = SEARCH_SAMPLE_TARGET  % len(search_configs)
group_targets = {
    cfg: base_group_target + (1 if idx < extra else 0)
    for idx, cfg in enumerate(search_configs)
}

print(f"Generating {SEARCH_SAMPLE_TARGET} diverse samples across "
      f"{len(search_configs)} groups (logging every {LOG_BATCH}).")

search_samples   = []
search_m_heights = []
rng_scatter      = np.random.default_rng(seed=9999)
last_logged      = 0
total_start      = time.time()

for cfg_index, (SEARCH_K, SEARCH_M) in enumerate(search_configs):
    target_count = group_targets[(SEARCH_K, SEARCH_M)]
    n_search  = int(target_count * SEARCH_FRACTION)
    n_scatter = target_count - n_search
    sigma     = sigma_map[(SEARCH_K, SEARCH_M)]
    sigma_mut = max(1.0, 0.15 * sigma)
    gens      = SEARCH_GENERATIONS + _extra_gens.get((SEARCH_K, SEARCH_M), 0)

    group_samples = []
    group_heights = []
    seen_group    = set()

    # ── Part A: Evolutionary search (low-m-height enrichment) ─────────────
    group_scored = []
    run_idx      = 0
    while len(group_samples) < n_search:
        if run_idx >= SEARCH_MAX_RUNS:
            break
        run_seed = SEARCH_BASE_SEED + cfg_index * 100 + run_idx
        scored_pool, _ = search_codes(
            n=SEARCH_N, k=SEARCH_K, target_ms=[SEARCH_M],
            seed=run_seed,
            pop_size=SEARCH_POP_SIZE, generations=gens,
            n_mut=int(0.9 * SEARCH_POP_SIZE), n_cross=int(0.5 * SEARCH_POP_SIZE),
            sigma_init=sigma, sigma_mut=sigma_mut,
            n_workers=SEARCH_WORKERS, clip_value=SEARCH_CLIP_VALUE, integerize=True,
        )
        group_scored.extend(scored_pool)
        group_scored.sort(key=lambda x: x["scalar"])
        for cand in group_scored:
            P   = np.clip(np.round(cand['G'][:, SEARCH_K:]),
                          -SEARCH_CLIP_VALUE, SEARCH_CLIP_VALUE).astype(int)
            key = tuple(P.ravel().tolist())
            if key in seen_group:
                continue
            seen_group.add(key)
            group_samples.append([SEARCH_N, SEARCH_K, SEARCH_M, P.copy()])
            group_heights.append(float(cand['scores'][SEARCH_M]))

            total_so_far = len(search_samples) + len(group_samples)
            if total_so_far - last_logged >= LOG_BATCH:
                elapsed = time.time() - total_start
                print(f"  [{total_so_far:>6}/{SEARCH_SAMPLE_TARGET}] "
                      f"{elapsed:.0f}s elapsed  |  "
                      f"k={SEARCH_K}, m={SEARCH_M} (search)")
                sys.stdout.flush()
                last_logged = total_so_far

            if len(group_samples) >= n_search:
                break
        run_idx += 1

    # ── Part B: Random scatter (full distribution) ────────────────────────
    scatter_items    = []
    scatter_rejected = 0
    while len(scatter_items) < n_scatter:
        if rng_scatter.random() < APC_FRACTION:
            P = gen_apc_P(SEARCH_K, SEARCH_N, rng_scatter, sigma)
        else:
            P = gen_gaussian_P(SEARCH_K, SEARCH_N, rng_scatter, sigma)
        d = min_distance(P, SEARCH_K, SEARCH_N)
        if SEARCH_M < d:
            key = tuple(P.ravel().tolist())
            if key not in seen_group:
                seen_group.add(key)
                scatter_items.append([SEARCH_N, SEARCH_K, SEARCH_M, P.copy()])
        else:
            scatter_rejected += 1

    scatter_work = [(i, s[0], s[1], s[2], s[3]) for i, s in enumerate(scatter_items)]
    scatter_h    = [None] * len(scatter_items)
    with Pool(SEARCH_WORKERS) as pool:
        for idx, height in pool.imap_unordered(_worker, scatter_work, chunksize=20):
            scatter_h[idx] = height

            total_so_far = len(search_samples) + len(group_samples) + (idx + 1)
            if total_so_far - last_logged >= LOG_BATCH:
                elapsed = time.time() - total_start
                print(f"  [{total_so_far:>6}/{SEARCH_SAMPLE_TARGET}] "
                      f"{elapsed:.0f}s elapsed  |  "
                      f"k={SEARCH_K}, m={SEARCH_M} (scatter)")
                sys.stdout.flush()
                last_logged = total_so_far

    for i, s in enumerate(scatter_items):
        group_samples.append(s)
        group_heights.append(scatter_h[i])

    search_samples.extend(group_samples)
    search_m_heights.extend(group_heights)

# ── Final summary ─────────────────────────────────────────────────────────
elapsed = time.time() - total_start
h_all   = np.array(search_m_heights)
print(f"\nDone in {elapsed:.1f}s — {len(search_samples)} samples total")
print(f"Height range: [{h_all.min():.2f}, {h_all.max():.2f}]  "
      f"median={np.median(h_all):.2f}")
print(f"\nPer-group summary:")
for grp in search_configs:
    idxs = [i for i, s in enumerate(search_samples) if (s[1], s[2]) == grp]
    h_g  = np.array([search_m_heights[i] for i in idxs])
    print(f"  k={grp[0]},m={grp[1]}: n={len(h_g):5d}  "
          f"min={h_g.min():.2f}  median={np.median(h_g):.2f}  "
          f"max={h_g.max():.2f}")

In [ ]:
# Save search-generated tuples and m-heights
search_out_dir = '/workspace/Homework/Project1'

with open(f'{search_out_dir}/Search_Generated_n_k_m_P', 'wb') as f:
    pickle.dump(search_samples, f)

with open(f'{search_out_dir}/Search_Generated_m-heights', 'wb') as f:
    pickle.dump(search_m_heights, f)

print(f"Saved {len(search_samples)} search samples to Search_Generated_n_k_m_P")
print(f"Saved {len(search_m_heights)} search m-heights to Search_Generated_m-heights")

# Verify by reloading
with open(f'{search_out_dir}/Search_Generated_n_k_m_P', 'rb') as f:
    check_search_data = pickle.load(f)
with open(f'{search_out_dir}/Search_Generated_m-heights', 'rb') as f:
    check_search_heights = pickle.load(f)

print(f"Verification: {len(check_search_data)} samples, {len(check_search_heights)} heights")
if len(check_search_heights) > 0:
    print(f"Heights range: [{min(check_search_heights):.6f}, {max(check_search_heights):.6f}]")
